# 🌱 OptiCrop — Smart Agricultural Production Optimization Engine

Machine-learning workflow for **crop recommendation** from soil & climate data.

**Pipeline:** load → EDA (univariate / bivariate / multivariate) → null check →
outlier analysis → seasonal crops → train/test split → K-Means clustering →
train & compare **KNN, Logistic Regression, Decision Tree, Random Forest** →
evaluate → save the best model with `pickle`.

**Features:** N, P, K, temperature, humidity, ph, rainfall &nbsp;→&nbsp; **Target:** label (22 crops)

## 1. Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle

import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('fivethirtyeight')
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print('Libraries imported.')

## 2. Read the Dataset

In [ ]:
# Works whether the notebook is run from notebooks/ or the project root
csv_path = '../dataset/Crop_recommendation.csv'
if not os.path.exists(csv_path):
    csv_path = 'dataset/Crop_recommendation.csv'

df = pd.read_csv(csv_path)
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nCrops:', df['label'].nunique())
df['label'].value_counts()

## 3. Univariate Analysis
Distribution of each individual feature.

In [ ]:
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.flatten(), features):
    sns.histplot(df[col], kde=True, ax=ax, color='#2e7d32')
    ax.set_title(f'Distribution of {col}')
for ax in axes.flatten()[len(features):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Count of samples per crop (balanced dataset)
plt.figure(figsize=(12, 5))
sns.countplot(y='label', data=df, color='#43a047',
              order=df['label'].value_counts().index)
plt.title('Number of samples per crop')
plt.tight_layout()
plt.show()

## 4. Bivariate Analysis
Relationship between humidity and crop label.

In [ ]:
plt.figure(figsize=(12, 7))
sns.boxplot(x='humidity', y='label', data=df,
            order=sorted(df['label'].unique()))
plt.title('Humidity vs Crop')
plt.tight_layout()
plt.show()

## 5. Multivariate Analysis
Correlation between the numeric features.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df[features].corr(), annot=True, cmap='YlGn', fmt='.2f',
            linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 6. Checking for Null Values

In [ ]:
print('Shape:', df.shape)
print('\n--- info ---')
df.info()
print('\n--- null counts ---')
print(df.isnull().sum())

## 7. Handling Outliers
Outliers are detected with boxplots and the **Interquartile Range (IQR)**:
`Upper = Q3 + 1.5*IQR`, `Lower = Q1 - 1.5*IQR`. Potassium (K) is right-skewed,
so a **log transform** is demonstrated to normalise its distribution.

In [ ]:
plt.figure(figsize=(13, 6))
sns.boxplot(data=df[features], palette='YlGn')
plt.title('Boxplots — outlier check')
plt.tight_layout()
plt.show()

# IQR outlier count per feature
for col in features:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n = int(((df[col] < lo) | (df[col] > hi)).sum())
    print(f'{col:<12}: {n} potential outlier(s)')

In [ ]:
# Log transform demonstration for Potassium (K)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['K'], kde=True, ax=ax[0], color='#795548')
ax[0].set_title('K — original (right-skewed)')
sns.histplot(np.log1p(df['K']), kde=True, ax=ax[1], color='#2e7d32')
ax[1].set_title('K — log1p transformed')
plt.tight_layout()
plt.show()

# NOTE: the deployed model (train_model.py) uses the raw features + StandardScaler
# so that training and the app's inference path stay identical (no train/serve skew).

## 8. Extracting Seasonal Crops
Group crops by the environmental conditions they prefer.

In [ ]:
summer = df[(df['temperature'] > 30) & (df['humidity'] < 60)]['label'].unique()
rainy  = df[(df['rainfall'] > 200) & (df['humidity'] > 80)]['label'].unique()
winter = df[(df['temperature'] < 20) & (df['humidity'] > 30)]['label'].unique()

print('🌞 Summer crops (hot, drier):', sorted(summer))
print('🌧️  Rainy crops (wet, humid):', sorted(rainy))
print('❄️  Winter crops (cooler):   ', sorted(winter))

## 9. Split into Train and Test Sets

In [ ]:
X = df[features]
y = df['label'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train.values)
X_test_sc = scaler.transform(X_test.values)
print('Train:', X_train.shape[0], '| Test:', X_test.shape[0])

## 10. K-Means Clustering (Elbow Method)
Unsupervised grouping of soil/climate patterns. The elbow point suggests a
natural number of clusters. (This is exploratory — the deployed recommender is
a supervised classifier.)

In [ ]:
wcss = []
K = range(1, 11)
for k in K:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_train_sc)
    wcss.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(K), wcss, marker='o', color='#2e7d32')
plt.title('K-Means Elbow Method')
plt.xlabel('Number of clusters (k)')
plt.ylabel('WCSS')
plt.tight_layout()
plt.show()

## 11. Model Building & Comparison
Train four classifiers and compare their accuracy.

In [ ]:
models = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

results = {}
trained = {}
for name, clf in models.items():
    clf.fit(X_train_sc, y_train)
    pred = clf.predict(X_test_sc)
    acc = accuracy_score(y_test, pred)
    results[name] = acc
    trained[name] = clf
    print(f'{name:<22} accuracy = {acc*100:.2f}%')

In [ ]:
# Accuracy comparison
plt.figure(figsize=(8, 4))
names = list(results.keys())
accs = [results[n] * 100 for n in names]
sns.barplot(x=accs, y=names, palette='YlGn')
plt.xlim(0, 100)
plt.xlabel('Accuracy (%)')
plt.title('Classifier Accuracy Comparison')
plt.tight_layout()
plt.show()

## 12. Evaluate the Best Model

In [ ]:
best_name = max(results, key=results.get)
best_model = trained[best_name]
print(f'Best model: {best_name} ({results[best_name]*100:.2f}%)\n')

pred = best_model.predict(X_test_sc)
print('Confusion matrix shape:', confusion_matrix(y_test, pred).shape)
print('\nClassification report:\n')
print(classification_report(y_test, pred))

## 13. Save the Best Model with Pickle

In [ ]:
models_dir = '../models' if os.path.basename(os.getcwd()) == 'notebooks' else 'models'
os.makedirs(models_dir, exist_ok=True)

with open(os.path.join(models_dir, 'model.pkl'), 'wb') as f:
    pickle.dump(best_model, f)
with open(os.path.join(models_dir, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

print('Saved model.pkl and scaler.pkl to', models_dir)

## 14. Predict a New Sample

In [ ]:
# N, P, K, temperature, humidity, ph, rainfall
sample = np.array([[90, 42, 43, 20.87, 82.0, 6.5, 202.9]])
sample_sc = scaler.transform(sample)
proba = best_model.predict_proba(sample_sc)[0]
classes = best_model.classes_
order = np.argsort(proba)[::-1]

print('Recommended crop:', classes[order[0]], f'({proba[order[0]]*100:.1f}% confidence)')
print('Top 3:', [(classes[i], round(float(proba[i])*100, 1)) for i in order[:3]])